# 01 · Data Ingestion
### *Stage 1 — acquire the five tables, validate the schema, log provenance*

> **Governing frame (from [Stage 0](../IMPLEMENTATION_PLAN.md)):** Fraud here is generated by a known
> formula, so this project measures *methodology quality*, not detection bragging rights. Ingestion's
> job is to confirm we have exactly the data the DOCS describe — nothing surprising, nothing missing.

**Gate for this stage:** raw untouched · counts logged · schema matches · PII review (synthetic ⇒ N/A).

In [1]:
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 60)
RAW = Path("../datasets")          # treat as data/raw — immutable
PROC = Path("../data/processed"); PROC.mkdir(parents=True, exist_ok=True)

def file_hash(p, blocksize=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(blocksize), b""):
            h.update(chunk)
    return h.hexdigest()[:12]

print("Ingestion run:", pd.Timestamp.now().isoformat(timespec="seconds"))

Ingestion run: 2026-07-19T19:16:54


#### Load all five tables (parse the two datetime columns on read)

In [2]:
transactions = pd.read_csv(RAW / "transactions.csv", parse_dates=["timestamp"])
accounts     = pd.read_csv(RAW / "account_profiles.csv")
patterns     = pd.read_csv(RAW / "fraud_patterns.csv")
edges        = pd.read_csv(RAW / "network_edges.csv")
timeseries   = pd.read_csv(RAW / "time_series_stats.csv", parse_dates=["hour"])

tables = {"transactions": transactions, "account_profiles": accounts,
          "fraud_patterns": patterns, "network_edges": edges, "time_series_stats": timeseries}
print("loaded", len(tables), "tables")

loaded 5 tables


#### Provenance log — source, row/col counts, and a content hash per file (Stage 1 best practice)

In [3]:
meta = []
for name, df in tables.items():
    p = RAW / f"{name}.csv"
    meta.append({"table": name, "rows": len(df), "cols": df.shape[1],
                 "mem_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
                 "md5_12": file_hash(p)})
meta = pd.DataFrame(meta)
meta

,table,rows,cols,mem_mb,md5_12
0,transactions,1000000,23,227.4,6cb22f0a06e6
1,account_profiles,50000,23,10.2,e61a2bc2b233
2,fraud_patterns,7,12,0.0,c74feaeead89
3,network_edges,7411,6,0.6,74f6c1d6bec7
4,time_series_stats,26280,10,2.1,e4bbde25f5ba


#### Schema & expectation validation
We assert the *specific* facts the DOCS promise. If any fails, ingestion stops here — a silent schema
drift is exactly what this gate exists to catch.

In [4]:
expected_cols = {
    "transactions": {"transaction_id","account_id","timestamp","hour_of_day","day_of_week","is_weekend",
        "amount","merchant_category","mcc_code","merchant_country","card_present","device_type",
        "device_known","ip_risk_score","is_foreign_txn","time_since_last_s","velocity_1h",
        "amount_vs_avg_ratio","account_age_days","has_2fa","credit_limit","is_fraud","fraud_pattern"},
}
for name, cols in expected_cols.items():
    missing = cols - set(tables[name].columns)
    assert not missing, f"{name} missing columns: {missing}"

n = len(transactions)
fraud_rate = transactions["is_fraud"].mean()
n_patterns = transactions.loc[transactions["is_fraud"] == 1, "fraud_pattern"].nunique()

assert n == 1_000_000,                 f"expected 1,000,000 txns, got {n:,}"
assert 0.016 <= fraud_rate <= 0.019,   f"fraud rate {fraud_rate:.4f} outside expected ~1.71%"
assert n_patterns == 7,                f"expected 7 fraud patterns, got {n_patterns}"
assert accounts["account_id"].nunique() == 50_000, "expected 50,000 accounts"

print(f"PASS · {n:,} transactions · fraud rate {fraud_rate:.4%} ({transactions['is_fraud'].sum():,} frauds)")
print(f"PASS · {n_patterns} fraud patterns · {accounts['account_id'].nunique():,} accounts · {len(edges):,} network edges")

PASS · 1,000,000 transactions · fraud rate 1.7143% (17,143 frauds)
PASS · 7 fraud patterns · 50,000 accounts · 7,411 network edges


#### Governance note
This is **synthetic** data (`numpy.random.default_rng(seed=42)`) — no real PII, so no masking or
consent review is required. In a production analogue, `account_id`, `merchant_country`, and device
identifiers would be classified and handled at this stage before any local copy is pulled.

---
### Stage 1 gate — ✅ cleared
- [x] Raw kept untouched in `../datasets/` · [x] source/counts/hash logged above
- [x] Schema matches the data dictionary · [x] PII review = N/A (synthetic)

**Next → [`02_cleaning.ipynb`](02_cleaning.ipynb):** enforce dtypes, run business-rule validation, and
write the leakage-safe processed tables.